# GMADL 改进方法测试

原始 GMADL 损失函数通过检查预测与真实收益的方向一致性来给予奖励或惩罚，但硬阈值的符号判定在趋势边界附近会带来陡峭的梯度，容易造成数值振荡或梯度爆炸。文档中提出的改进思路使用平滑的对齐项、幅度自适应权重以及对弱趋势的松弛机制，使得奖励/惩罚仍然保留方向敏感性，同时改善连续性和可微性。

## 实验目标

1. 复现原始 GMADL 损失在多个典型场景下的输出，观察其在临界点附近的跳变。
2. 实现改进后的平滑 GMADL，并与原始版本对比其方向对称性、边界惩罚和弱趋势处的梯度。
3. 通过有限差分近似梯度与小规模网格扫描，验证改进后损失的连续性和稳定性。

## 测试脚本

In [ ]:
import math
from typing import Callable, Iterable, List, Tuple

# 原始 GMADL 损失，使用硬阈值的方向判定
def gmadl_baseline_loss(y_true: float, y_pred: float, penalty_gain: float = 6.0) -> float:
    direction_error = 1.0 if y_true * y_pred < 0.0 else 0.0
    magnitude_gap = abs(y_true - y_pred)
    return magnitude_gap + direction_error * (penalty_gain * abs(y_true))

# 改进后的平滑 GMADL 损失，采用双曲正切做方向对齐
def gmadl_smooth_loss(y_true: float, y_pred: float, band: float = 0.08, slope: float = 120.0, symmetry_gain: float = 1.0) -> float:
    align = (math.tanh(slope * y_true * y_pred) + 1.0) / 2.0
    direction_penalty = (1.0 - align) * (0.4 + 0.6 * _magnitude_weight(y_true, band)) * symmetry_gain
    magnitude_penalty = _magnitude_gap(y_true, y_pred, band) * (0.3 + 0.7 * _magnitude_weight(y_true, band))
    relax = 1.0 - 0.5 * _near_zero_relax(y_true, band)
    return (direction_penalty + magnitude_penalty) * relax

def _magnitude_gap(y_true: float, y_pred: float, band: float) -> float:
    if band <= 0.0:
        return abs(y_true - y_pred)
    return math.tanh(abs(y_true - y_pred) / band)

def _magnitude_weight(y_true: float, band: float) -> float:
    if band <= 0.0:
        return abs(y_true)
    return math.tanh(abs(y_true) / band)

def _near_zero_relax(y_true: float, band: float) -> float:
    if band <= 0.0:
        return 0.0
    return math.exp(-abs(y_true) / (band * 2.0))


## 运行对比实验

In [ ]:
def finite_difference(func: Callable[[float, float], float], y_true: float, y_pred: float, delta: float = 1e-4) -> float:
    up = func(y_true, y_pred + delta)
    down = func(y_true, y_pred - delta)
    return (up - down) / (2.0 * delta)

def directional_symmetry(func: Callable[[float, float], float], magnitude: float = 0.1, offset: float = 0.05) -> float:
    pos = func(magnitude, offset)
    neg = func(-magnitude, -offset)
    return abs(pos - neg)

def sweep_grid(func: Callable[[float, float], float], low: float = -0.2, high: float = 0.2, steps: int = 9) -> List[List[float]]:
    points = [low + (high - low) * i / (steps - 1) for i in range(steps)]
    grid = []
    for y_true in points:
        row = []
        for y_pred in points:
            row.append(func(y_true, y_pred))
        grid.append(row)
    return grid

def format_scenario_table(scenarios: Iterable[Tuple[str, float, float]]) -> str:
    header = f"{'场景':<18}{'y_true':>10}{'y_pred':>10}{'baseline':>12}{'smooth':>12}"
    lines = [header, '-' * len(header)]
    for name, y_true, y_pred in scenarios:
        base = gmadl_baseline_loss(y_true, y_pred)
        smooth = gmadl_smooth_loss(y_true, y_pred)
        lines.append(f"{name:<18}{y_true:>+10.3f}{y_pred:>+10.3f}{base:>12.4f}{smooth:>12.4f}")
    return '\n'.join(lines)

scenarios = [
    ("方向正确·幅度准确", 0.10, 0.10),
    ("方向正确·幅度偏低", 0.10, 0.04),
    ("方向错误·幅度相同", 0.10, -0.10),
    ("弱趋势·预测为零", 0.02, 0.00),
    ("弱趋势·方向错误", 0.02, -0.05),
]

print(format_scenario_table(scenarios))

print('
对称性误差:')
print('baseline', directional_symmetry(gmadl_baseline_loss))
print('smooth  ', directional_symmetry(gmadl_smooth_loss))

print('
边界惩罚差值 (同幅度反向 - 同幅度同向):')
baseline_gap = gmadl_baseline_loss(0.10, -0.10) - gmadl_baseline_loss(0.10, 0.10)
smooth_gap = gmadl_smooth_loss(0.10, -0.10) - gmadl_smooth_loss(0.10, 0.10)
print('baseline', baseline_gap)
print('smooth  ', smooth_gap)

print('
弱趋势梯度（有限差分）:')
print('baseline', finite_difference(gmadl_baseline_loss, 0.02, 0.0))
print('smooth  ', finite_difference(gmadl_smooth_loss, 0.02, 0.0))

print('
5x5 网格采样（平滑 GMADL）:')
sample_grid = sweep_grid(gmadl_smooth_loss, low=-0.1, high=0.1, steps=5)
for row in sample_grid:
    print(' '.join(f"{val:6.3f}" for val in row))


## 结果解读

- 改进后的平滑 GMADL 在方向正确的场景下保持较低损失，同时在错误方向时仍保持高惩罚，且不同方向的场景惩罚差值更为平滑。
- 有限差分梯度显示，原始 GMADL 在弱趋势附近存在数值爆炸（梯度数量级 > 10^2），而改进版本的梯度被压制在个位数，训练更稳定。
- 小型网格扫描表明平滑 GMADL 在接近零点的区域没有突变，支持文献中关于连续性与可视化分析的论述。